# Data understanding

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)

### **load dataset**

In [38]:
DATA_PATH = "../data/raw/online_retail_II.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### **data set shape**

In [39]:
print("Number of records: ",df.shape[0])
print("Number of features: ",df.shape[1])

Number of records:  1067371
Number of features:  8


### **display columns**

In [40]:
print("Columns: ")

for column in df.columns:
    print("-",column)

Columns: 
- Invoice
- StockCode
- Description
- Quantity
- InvoiceDate
- Price
- Customer ID
- Country


### **Dataset Information**

In [41]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


### **Statistical summary**

In [42]:
df.describe(include="all").T
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Quantity,1067371.0,9.938898,172.705794,-80995.00,1.00,3.0,10.00,80995.0
Price,1067371.0,4.649388,123.553059,-53594.36,1.25,2.1,4.15,38970.0
Customer ID,824364.0,15324.638504,1697.464450,12346.00,13975.00,15255.0,16797.00,18287.0


By default, describe() summarizes numeric columns only.
transpose (.T) as flipping a table. (Flip the DataFrame's rows and columns)

### **Data types**

In [43]:
data_types = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values
})

data_types

,Column,Data Type
0,Invoice,str
1,StockCode,str
2,Description,str
3,Quantity,int64
4,InvoiceDate,str
5,Price,float64
6,Customer ID,float64
7,Country,str


### **Missing values**

In [44]:
missing_values = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})

missing_values

,Missing Count,Missing Percentage
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,243007,22.77
Country,0,0.00


mean(), returns decimal. Normally want to display it as a percentage. So do like "df.isnull().mean() * 100"
.round(2) keeps only 2 decimal places. (2.57346746 => 2.57)

### **Duplicate records**

In [45]:
duplicate_count = df.duplicated().sum()

print("Duplicate records:", duplicate_count)

Duplicate records: 34335


### **Unique values**

In [46]:
unique_values = pd.DataFrame({
    "Column": df.columns,
    "Unique Values": [df[column].nunique() for column in df.columns]
})

unique_values

,Column,Unique Values
0,Invoice,53628
1,StockCode,5305
2,Description,5698
3,Quantity,1057
4,InvoiceDate,47635
5,Price,2807
6,Customer ID,5942
7,Country,43


For every column in my dataset, count how many different values it contains and put the result into a table.
ex: 43 different countries. So nunique() = number of categories, while value_counts() = frequency of each category.

### **Date range**

In [47]:
temp_dates = pd.to_datetime(df["InvoiceDate"], errors="coerce")

print("Minimum date:", temp_dates.min())
print("Maximum date:", temp_dates.max())

Minimum date: 2009-12-01 07:45:00
Maximum date: 2011-12-09 12:50:00


errors="coerce" => If a value cannot be converted to a valid date, Pandas replaces it with NaT (Not a Time) instead of throwing an error. Use as safe conversion. As summary select InvoiceDate column and convert string date values into Pandas Timestamp objects. Store converted datetime values in a temporary variable without modifying the original DataFrame (df).

### **Negative quantities**

In [48]:
negative_quantity_count = (df["Quantity"] < 0).sum()

print("Negative quantity records:", negative_quantity_count)

Negative quantity records: 22950


Identify negative quantities as returns/cancellations.

### **Zero quantities**

In [49]:
zero_quantity_count = (df["Quantity"] == 0).sum()

print("Zero quantity records:", zero_quantity_count)

Zero quantity records: 0


Identify zero quantities as data entry mistake or system error. Normaly in E-commerce / Online Retail Data Analysis drop these zero quantity records.

### **Zero prices**

In [50]:
zero_price_count = (df["Price"] == 0).sum()

print("Zero price records:", zero_price_count)

Zero price records: 6202


### **negative prices**

In [51]:
negative_price_count = (df["Price"] < 0).sum()

print("Negative price records:", negative_price_count)

Negative price records: 5


### **Canceled invoics**

In [52]:
cancelled_invoice_count = (
    df["Invoice"]
    .astype(str)
    .str.startswith("C")
    .sum()
)

print("Cancelled invoice records:", cancelled_invoice_count)

Cancelled invoice records: 19494


### **create a single data quality summary**

In [53]:
data_quality_summary = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Total Features",
        "Missing Description",
        "Missing Customer ID",
        "Duplicate Records",
        "Negative Quantity",
        "Zero Quantity",
        "Zero Price",
        "Negative Price",
        "Cancelled Invoices",
        "Unique Customers",
        "Unique Products",
        "Unique Invoices",
        "Unique Countries"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df["Description"].isnull().sum(),
        df["Customer ID"].isnull().sum(),
        df.duplicated().sum(),
        (df["Quantity"] < 0).sum(),
        (df["Quantity"] == 0).sum(),
        (df["Price"] == 0).sum(),
        (df["Price"] < 0).sum(),
        df["Invoice"].astype(str).str.startswith("C").sum(),
        df["Customer ID"].nunique(),
        df["StockCode"].nunique(),
        df["Invoice"].nunique(),
        df["Country"].nunique()
    ]
})

data_quality_summary

,Metric,Value
0,Total Records,1067371
1,Total Features,8
2,Missing Description,4382
3,Missing Customer ID,243007
4,Duplicate Records,34335
5,Negative Quantity,22950
6,Zero Quantity,0
7,Zero Price,6202
8,Negative Price,5
9,Cancelled Invoices,19494


### **Save the understanding report**

In [54]:

output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

data_quality_summary.to_csv(
    "../data/processed/data_quality_summary.csv",
    index=False
)